In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling1D, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

np.random.seed(7)
tf.random.set_seed(7)



2026-05-06 11:31:41.986727: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778067101.999516      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778067102.003405      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-06 11:31:42.018737: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# load training data
df = pd.read_csv("../input/train.csv")
a2c = {"EAP": 0, "HPL": 1, "MWS": 2}
y = np.array([a2c[a] for a in df.author])
y = to_categorical(y)




In [3]:
def preprocess(text):
    text = text.replace("' ", " ' ")
    signs = set(',.:;"?!')
    prods = set(text) & signs
    if not prods:
        return text
    for sign in prods:
        text = text.replace(sign, f" {sign} ")
    return text




In [4]:
def create_docs(df, n_gram_max=2):
    def add_ngram(tokens, n_gram_max):
        ngrams = []
        for n in range(2, n_gram_max + 1):
            for i in range(len(tokens) - n + 1):
                ngrams.append("--".join(tokens[i : i + n]))
        return tokens + ngrams

    docs = []
    for txt in df.text:
        tokens = preprocess(txt).split()
        docs.append(" ".join(add_ngram(tokens, n_gram_max)))
    return docs




In [5]:
min_count = 2
docs = create_docs(df)

# first tokenizer to count word frequencies
tokenizer_tmp = Tokenizer(lower=False, filters="")
tokenizer_tmp.fit_on_texts(docs)
num_words = sum(1 for cnt in tokenizer_tmp.word_counts.values() if cnt >= min_count)

# final tokenizer limited to frequent words
tokenizer = Tokenizer(num_words=num_words, lower=False, filters="")
tokenizer.fit_on_texts(docs)
seqs = tokenizer.texts_to_sequences(docs)

maxlen = 256
docs_padded = pad_sequences(sequences=seqs, maxlen=maxlen)

input_dim = np.max(docs_padded) + 1  # vocabulary size (+1 for padding token)



In [6]:
x_train, x_val, y_train, y_val = train_test_split(
    docs_padded, y, test_size=0.20, random_state=42, stratify=y
)




In [7]:
def create_model(embedding_dims=20, optimizer="adam"):
    model = Sequential()
    model.add(Embedding(input_dim=int(input_dim), output_dim=embedding_dims))
    model.add(GlobalAveragePooling1D())
    model.add(Dense(3, activation="softmax"))
    model.compile(
        loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"]
    )
    return model


model = create_model()
model.summary()



2026-05-06 11:31:50.288526: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778067110.288996      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:45:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
epochs = 25
history = model.fit(
    x_train,
    y_train,
    batch_size=16,
    validation_data=(x_val, y_val),
    epochs=epochs,
    callbacks=[
        EarlyStopping(patience=2, monitor="val_loss", restore_best_weights=True)
    ],
    verbose=2,
)



Epoch 1/25


I0000 00:00:1778067111.001372     107 service.cc:148] XLA service 0x7f8018002f50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778067111.001413     107 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-06 11:31:51.014827: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778067111.052147     107 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1778067111.520482     107 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


881/881 - 3s - 3ms/step - accuracy: 0.4085 - loss: 1.0805 - val_accuracy: 0.4085 - val_loss: 1.0641
Epoch 2/25
881/881 - 1s - 1ms/step - accuracy: 0.4926 - loss: 1.0211 - val_accuracy: 0.6250 - val_loss: 0.9636
Epoch 3/25
881/881 - 1s - 1ms/step - accuracy: 0.6447 - loss: 0.8749 - val_accuracy: 0.6525 - val_loss: 0.8417
Epoch 4/25
881/881 - 1s - 1ms/step - accuracy: 0.7372 - loss: 0.7176 - val_accuracy: 0.6698 - val_loss: 0.7518
Epoch 5/25
881/881 - 1s - 1ms/step - accuracy: 0.7931 - loss: 0.5948 - val_accuracy: 0.7118 - val_loss: 0.6645
Epoch 6/25
881/881 - 1s - 1ms/step - accuracy: 0.8338 - loss: 0.4991 - val_accuracy: 0.7441 - val_loss: 0.5954
Epoch 7/25
881/881 - 1s - 1ms/step - accuracy: 0.8644 - loss: 0.4215 - val_accuracy: 0.7784 - val_loss: 0.5435
Epoch 8/25
881/881 - 1s - 1ms/step - accuracy: 0.8893 - loss: 0.3571 - val_accuracy: 0.7983 - val_loss: 0.5017
Epoch 9/25
881/881 - 1s - 1ms/step - accuracy: 0.9088 - loss: 0.3032 - val_accuracy: 0.8128 - val_loss: 0.4675
Epoch 10/25


In [9]:
# prepare test data
test_df = pd.read_csv("../input/test.csv")
test_docs = create_docs(test_df)
test_seqs = tokenizer.texts_to_sequences(test_docs)
test_padded = pad_sequences(sequences=test_seqs, maxlen=maxlen)

preds = model.predict(test_padded, batch_size=256)



8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step


In [10]:
submission = pd.read_csv("../input/sample_submission.csv")
for author, idx in a2c.items():
    submission[author] = preds[:, idx]

submission.to_csv("fastText_result_04.csv", index=False)